# Migrate CFQuality, CFROI, CFKeypoints to Attribute System

This notebook migrates the ImageInstance columns `CFQuality`, `CFROI`, and `CFKeypoints` to the attribute system.

The migration is **idempotent** - it can be run multiple times without duplicating data. Existing AttributeValues are updated rather than duplicated.

## Steps:
1. **Setup**: Create database session and import dependencies
2. **Create Attributes & Models**: Create AttributeDefinitions and AttributesModels for each attribute, linking them together
3. **Load Image IDs**: Get all ColorFundus ImageInstance IDs to process
4. **Preload Existing Values**: Load existing AttributeValues into memory for efficient lookups
5. **Batch Migration**: Process images in batches (5000 at a time), creating/updating AttributeValue records for each attribute



In [4]:
# Setup: Import dependencies and create database session
from eyened_orm import Database
from eyened_orm import ImageInstance
from eyened_orm.attributes import (
    AttributeDefinition,
    AttributeDataType,
    AttributeValue,
    AttributesModel,
    AttributesModelOutput,
)
from eyened_orm.segmentation import ModelKind
from sqlalchemy import select, func
from sqlalchemy.orm import Session
from tqdm import tqdm

# database = Database('../../dev/.env')
database = Database('/home/bart/25/eyened-platform/environments/prod_wr.env')

session = database.create_session()

## Helper Functions

Helper functions for idempotent creation of AttributeDefinitions, AttributesModels, and links.


In [5]:
def get_or_create_attribute_definition(
    session: Session, attribute_name: str, data_type: AttributeDataType
) -> AttributeDefinition:
    return AttributeDefinition.get_or_create(
        session,
        match_by={"AttributeName": attribute_name},
        create_kwargs={"AttributeName": attribute_name, "AttributeDataType": data_type},
        must_match={"AttributeDataType": data_type},
        verbose=True,
    )


def get_or_create_attributes_model(
    session: Session, model_name: str, version: str, description: str
) -> AttributesModel:
    return AttributesModel.get_or_create(
        session,
        match_by={
            "ModelName": model_name,
            "Version": version,
            "ModelType": ModelKind.Attributes,
        },
        create_kwargs={
            "ModelName": model_name,
            "Version": version,
            "Description": description,
            "ModelType": ModelKind.Attributes,
        },
        verbose=True,
    )


def get_or_create_model_output_link(
    session: Session, model_id: int, attribute_id: int
) -> AttributesModelOutput:
    return AttributesModelOutput.get_or_create(
        session,
        match_by={"ModelID": model_id, "AttributeID": attribute_id},
        verbose=True,
    )

## Step 1: Create AttributeDefinitions and Models

Create AttributeDefinitions and AttributesModels for the three attributes, linking them together. Each attribute gets:
- **cf_quality**: Float type, linked to `cf_quality_legacy` model
- **cf_roi**: JSON type, linked to `cf_roi_legacy` model  
- **cf_keypoints**: JSON type, linked to `cf_keypoints_legacy` model


In [6]:
def get_or_create_attr_and_model(session, attr_name, data_type, model_name, description):
    attr = get_or_create_attribute_definition(session, attr_name, data_type)
    model = get_or_create_attributes_model(
        session,
        model_name=model_name,
        version="1.0",
        description=description
    )
    if attr not in model.OutputAttributes:
        model.OutputAttributes.append(attr)
    return attr, model

# quality
cf_quality_attr, cf_quality_model = get_or_create_attr_and_model(
    session,
    "cf_quality",
    AttributeDataType.Float,
    "cf_quality_legacy",
    "Legacy CFQuality column migration"
)

# roi
cf_roi_attr, cf_roi_model = get_or_create_attr_and_model(
    session,
    "cf_roi",
    AttributeDataType.JSON,
    "cf_roi_legacy",
    "Legacy CFROI column migration"
)

# keypoints
cf_keypoints_attr, cf_keypoints_model = get_or_create_attr_and_model(
    session,
    "cf_keypoints",
    AttributeDataType.JSON,
    "cf_keypoints_legacy",
    "Legacy CFKeypoints column migration"
)

session.commit()

Created AttributeDefinition: AttributeDefinition(AttributeID=7, AttributeName=cf_quality, AttributeDataType=AttributeDataType.Float)
Created AttributesModel: AttributesModel(ModelID=28)
Created AttributeDefinition: AttributeDefinition(AttributeID=8, AttributeName=cf_roi, AttributeDataType=AttributeDataType.JSON)
Created AttributesModel: AttributesModel(ModelID=29)
Created AttributeDefinition: AttributeDefinition(AttributeID=9, AttributeName=cf_keypoints, AttributeDataType=AttributeDataType.JSON)
Created AttributesModel: AttributesModel(ModelID=30)


## Load Image IDs

Get all ColorFundus ImageInstance IDs that need to be processed. This determines the scope of the migration.


In [7]:
# Get all color fundus ImageInstances
from eyened_orm import Modality
image_ids = session.scalars(
    select(ImageInstance.ImageInstanceID).filter(
        ImageInstance.Modality == Modality.ColorFundus
    )
).all()
print(f"\nTotal ImageInstances to process: {len(image_ids)}")


Total ImageInstances to process: 751842


## Step 2: Preload Existing AttributeValues

Load existing AttributeValues into memory for efficient lookups during migration. This ensures the migration is idempotent - existing records are updated rather than duplicated.


In [8]:
# Preload existing AttributeValues for efficiency
# Query existing AttributeValues for the images and attributes we'll be processing
existing_av_map = {}

existing_avs = session.scalars(
    select(AttributeValue).where(
        AttributeValue.ImageInstanceID.in_(image_ids),
        AttributeValue.AttributeID.in_(
            [
                cf_quality_attr.AttributeID,
                cf_roi_attr.AttributeID,
                cf_keypoints_attr.AttributeID,
            ]
        ),
        AttributeValue.ModelID.in_(
            [cf_quality_model.ModelID, cf_roi_model.ModelID, cf_keypoints_model.ModelID]
        ),
    )
).all()

# Create lookup dict: (ImageInstanceID, AttributeID, ModelID) -> AttributeValue
existing_av_map = {
    (av.ImageInstanceID, av.AttributeID, av.ModelID): av for av in existing_avs
}
print(f"Found {len(existing_av_map)} existing AttributeValues")

Found 0 existing AttributeValues


## Step 3: Batch Migration

Migrate values from ImageInstance columns to AttributeValue records in batches of 5000 images. 

The migration uses a configuration-driven approach:
- Each attribute has a value extractor function (lambda) that extracts the appropriate field from ImageInstance
- Images are processed in batches to manage memory
- Existing AttributeValues are updated, new ones are created as needed
- Progress is tracked with tqdm progress bars


In [9]:
BATCH_SIZE = 5000

def get_or_create_av(session: Session, image_id: int, attribute_id: int, model_id: int,
                     value_float=None, value_json=None, existing_map=None):
    """Get existing or create new AttributeValue."""
    key = (image_id, attribute_id, model_id)
    av = existing_map.get(key) if existing_map else None
    if not av:
        av = AttributeValue(
            ImageInstanceID=image_id,
            AttributeID=attribute_id,
            ModelID=model_id
        )
        session.add(av)
    av.ValueFloat = value_float
    av.ValueJSON = value_json
    return av


def process_attribute_in_batches(image_ids, attr, model, session, existing_map, get_value):
    """Process images in batches, creating/updating AttributeValues."""
    attr_name = getattr(attr, 'AttributeName', str(attr))
    for i in tqdm(range(0, len(image_ids), BATCH_SIZE), desc=f"Processing {attr_name}"):
        batch_ids = image_ids[i:i + BATCH_SIZE]
        for image in ImageInstance.by_ids(session, batch_ids):
            value_fields = get_value(image)
            if value_fields:
                av = get_or_create_av(
                    session, image.ImageInstanceID, attr.AttributeID, model.ModelID,
                    existing_map=existing_map, **value_fields
                )
                existing_map[(av.ImageInstanceID, av.AttributeID, av.ModelID)] = av
        session.commit()


# Migration config: (attr, model, value_getter)
migrations = [
    (cf_quality_attr, cf_quality_model, lambda img: {'value_float': img.CFQuality} if img.CFQuality else None),
    (cf_roi_attr, cf_roi_model, lambda img: {'value_json': img.CFROI} if img.CFROI else None),
    (cf_keypoints_attr, cf_keypoints_model, lambda img: {'value_json': img.CFKeypoints} if img.CFKeypoints else None),
]

for attr, model, get_value in migrations:
    process_attribute_in_batches(image_ids, attr, model, session, existing_av_map, get_value)




Processing cf_quality:   0%|          | 0/151 [00:00<?, ?it/s]

Processing cf_keypoints: 100%|██████████| 151/151 [1:07:07<00:00, 26.67s/it]
